# Plant Diseases Classification — Google Colab Runner

Trains the model on Colab's free GPU (T4), and also produces a small subset of the dataset that you can download to your laptop for fast local development under DVC.

## Before you start

1. **Runtime → Change runtime type → GPU.** Confirmed in the next cell.
2. **Kaggle API token:**
   - https://www.kaggle.com/settings → API → Create New API Token.
   - Upload the downloaded `kaggle.json` to your Drive at `My Drive/kaggle/kaggle.json`.

Expected total runtime: 30–60 min for 5 full epochs on T4.

## 1. Verify GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Mount Google Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 3. Clone the project repo

In [ ]:
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/ri377/plant-diseases-classification.git"
REPO_BRANCH = "main"
REPO_DIR = Path("/content/plant-diseases-classification")

if REPO_DIR.exists():
    print(f"{REPO_DIR} already exists — pulling latest")
    subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)
else:
    subprocess.run(
        ["git", "clone", "--branch", REPO_BRANCH, "--depth", "1", REPO_URL, str(REPO_DIR)],
        check=True,
    )

%cd {REPO_DIR}
!ls

## 4. Install the project

In [ ]:
!pip install --quiet uv
!uv pip install --system --quiet -e .

## 5. Set up Kaggle credentials

In [ ]:
import shutil
from pathlib import Path

KAGGLE_JSON_SRC = Path("/content/drive/MyDrive/kaggle/kaggle.json")
KAGGLE_DIR = Path.home() / ".kaggle"
KAGGLE_DIR.mkdir(parents=True, exist_ok=True)
shutil.copy(KAGGLE_JSON_SRC, KAGGLE_DIR / "kaggle.json")
(KAGGLE_DIR / "kaggle.json").chmod(0o600)
print("Kaggle credentials installed.")

## 6. Download the dataset (full)

Downloads ~3 GB to `data/new-plant-diseases-dataset/` (matches `data.root` in the config). Takes ~3 minutes.

In [ ]:
import os

os.environ["KAGGLE_CONFIG_DIR"] = str(Path.home() / ".kaggle")

!mkdir -p data/new-plant-diseases-dataset
!kaggle datasets download -d vipoooool/new-plant-diseases-dataset \
    -p data/new-plant-diseases-dataset --unzip
!ls 'data/new-plant-diseases-dataset/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)' | head

## 7. Make a small subset and save it to Drive

Creates ~50 MB of subset data (10 random classes × ~38 images) and copies it into your Drive. Download this folder from Drive to your laptop, then `dvc add` + `dvc push` it locally.

Tune the size by changing `classes_per_subset`, `train_per_class`, `val_per_class`.

In [ ]:
!python scripts/make_subset.py \
    --source data/new-plant-diseases-dataset \
    --output /content/plant-diseases-subset \
    --classes_per_subset 10 \
    --train_per_class 30 \
    --val_per_class 8

!du -sh /content/plant-diseases-subset

In [ ]:
import shutil
from pathlib import Path

DRIVE_SUBSET = Path("/content/drive/MyDrive/plant-diseases-classification/subset")
if DRIVE_SUBSET.exists():
    shutil.rmtree(DRIVE_SUBSET)
shutil.copytree("/content/plant-diseases-subset", DRIVE_SUBSET)
print(f"Subset copied to {DRIVE_SUBSET}")

## 8. Smoke test

Verifies the pipeline runs end-to-end (~1 min on GPU).

In [ ]:
!python commands.py train training=smoke_test

## 9. Full training

Adjust epochs from the CLI: `training.max_epochs=10`.

In [ ]:
!python commands.py train

## 10. Save run outputs to Drive

In [ ]:
import shutil
from datetime import datetime
from pathlib import Path

stamp = datetime.now().strftime("%Y%m%d-%H%M%S")
OUT_DIR = Path(f"/content/drive/MyDrive/plant-diseases-classification/runs/{stamp}")
OUT_DIR.mkdir(parents=True, exist_ok=True)

for sub in ["checkpoints", "plots", "artifacts", "logs"]:
    src = Path("/content/plant-diseases-classification") / sub
    if src.exists():
        shutil.copytree(src, OUT_DIR / sub, dirs_exist_ok=True)
        print(f"Copied {src} -> {OUT_DIR / sub}")

print(f"\nAll outputs saved under: {OUT_DIR}")

## After the run

On your Drive you'll find two things:

- `plant-diseases-classification/subset/` — small subset to download to your laptop for DVC tracking.
- `plant-diseases-classification/runs/<timestamp>/` — checkpoints, plots, logs, class mapping.

Next steps locally:

1. Download the `subset/` folder to your laptop, e.g. into `data/new-plant-diseases-dataset/`.
2. `uv run dvc add data/new-plant-diseases-dataset`
3. `uv run dvc push -r data-storage`
4. From now on `source=dvc` works for smoke tests anywhere.
5. Download the best checkpoint, add it to DVC under the `models-storage` remote.